In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [3]:
# load environment variables from .env file
load_dotenv()

True

In [4]:
# Define the persistent directory for Chroma database
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma vector store with the specified persistent directory and embedding model
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7535.29it/s]


In [5]:
# Set up AI model
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

# Store our conversation as messages
chat_history = []


def ask_question(user_question):
    print(f"\n--- You asked: {user_question} ---")

    # Step 1: Make the question clear using conversation history
    if chat_history:
        # Ask AI to make the question standalone
        messages = (
            [
                SystemMessage(
                    content="Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question."
                ),
            ]
            + chat_history
            + [HumanMessage(content=f"New question: {user_question}")]
        )

        result = model.invoke(messages)
        search_question = result.content.strip()
        print(f"Searching for: {search_question}")
    else:
        search_question = user_question

    # Step 2: Find relevant documents
    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(search_question)

    print(f"Found {len(docs)} relevant documents:")
    for i, doc in enumerate(docs, 1):
        # Show first 2 lines of each document
        lines = doc.page_content.split("\n")[:2]
        preview = "\n".join(lines)
        print(f"  Doc {i}: {preview}...")

    ## Step 3: Create final prompt
    combined_input = f"""Based on the following documents, please answer this question: {user_question}

    Documents:
    {chr(10).join([f"- {doc.page_content}" for doc in docs])}

    Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
    """

    # Step 4: Get the answer
    messages = (
        [
            SystemMessage(
                content="You are a helpful assistant that answers questions based on provided documents and conversation history."
            ),
        ]
        + chat_history
        + [HumanMessage(content=combined_input)]
    )

    result = model.invoke(messages)
    answer = result.content

    # Step 5: Remember this conversation
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=answer))

    print(f"Answer: {answer}")
    return answer

In [6]:
# Simple chat loop
def start_chat():
    print("Ask me questions! Type 'quit' to exit.")

    while True:
        question = input("\nYour question: ")

        if question.lower() == "quit":
            print("Goodbye!")
            break

        ask_question(question)

In [8]:
# Start the chat with history-aware generation
start_chat()

Ask me questions! Type 'quit' to exit.

--- You asked: When was Google founded? ---
Searching for: When was Google founded?
Found 3 relevant documents:
  Doc 1: 40. Hanley, Rachael (February 12, 2003). "From Googol to Google" (https://web.archive.org/we
b/20100327141327/http://www.stanforddaily.com/2003/02/12/from-googol-to-google). The...
  Doc 2: Larry Page and Sergey Brin in 2003 Stanford University in California, USA.[23][24][25] The project
initially involved an unofficial "third founder", Scott Hassan,...
  Doc 3: 42. Long, Tony (September 7, 2007). "Sept. 7, 1998: If the Check Says 'Google Inc.,' We're
'Google Inc.' " (https://www.wired.com/2007/09/dayintech-0907/). Wired. Archived (https://we...
Answer: Google was founded on September 7, 1998, as "Google Inc." [42].

--- You asked: Who were the founders? ---
Searching for: Who were the founders of Google?
Found 3 relevant documents:
  Doc 1: Larry Page and Sergey Brin in 2003 Stanford University in California, USA.[23][24][25] 